# Reproducing Bakalar et al. (2023) from dandiset 001362

Paper: Bakalar D, Gavrilova O, Jiang SZ, Zhang H-Y, Roy S, Williams SK, Liu N, Wisser S, Usdin TB,
Eiden LE (2023). Constitutive and conditional deletion reveals distinct phenotypes driven by
developmental versus neurotransmitter actions of the neuropeptide PACAP. *J Neuroendocrinol*
35(11), e13286. [doi:10.1111/jne.13286](https://doi.org/10.1111/jne.13286). The claims file was
written from the NIH author manuscript in PubMed Central (PMC10620107) with its Supplemental
Methods, which describe the whole-brain c-Fos pipeline and its statistics.

Dataset: [DANDI:001362](https://dandiarchive.org/dandiset/001362/0.260923.2152), version
0.260923.2152. The paper cites no version (its data statement says data are available on request),
so the most recent published version is pinned. The only other published version, 0.260701.1900,
holds the same 765 assets with the same blobs (checked below); the later one adds ORCIDs and the
paper's DOI to the metadata, none of which the analysis uses.

What the dataset holds: a BIDS derivative dataset (not NWB) of the whole-brain c-Fos light-sheet
experiment of Figure 5 and Table 4 only: for 22 hemispheres of 13 mice (6 PACAP fl/fl, 7 PACAPko),
the deconvolved and N4-corrected stacks, the registered atlas labels, the FRST image, binary FRST
segmentations at several thresholds, atlas-space heatmaps, and, for 13 hemispheres (one per mouse),
per-region cell count tables. None of the paper's behaviour, qPCR, immunohistochemistry, in situ
hybridization, restraint or telemetry data is in it, and no other dandiset of this paper or lab was
found on the archive.

What is done here: cells are recounted per atlas region from the binary segmentations of all 22
hemispheres, by the counting rule of the Supplemental Methods (26-connected components of 9 to 900
voxels, each assigned to the label holding most of its voxels), at the five FRST thresholds the
segmentations of every hemisphere share, and at the finer series that 4 hemispheres hold. The
notebook `reproduce_bakalar_2023_thresholds_14000_26000.ipynb` runs thresholds 14000 to 26000 on the
18 hemispheres that hold them. The recount is checked against the dataset's own count
tables. The region-wise Mann-Whitney U tests with Benjamini-Krieger-Yekutieli adjustment are then
run under every reading the claims file names. The recount streams the segmentations and label
images of 22 hemispheres and took 52 minutes in the last run; nothing is written to disk. The
synthetic check of the counting draws its volume from a generator with a fixed seed.

| Claim | Data | Status |
|---|---|---|
| fig5-dg-fos | dataset | tested |
| table4-lenient-regions | dataset | tested (depends on fig5-dg-fos) |
| table4-region-values | dataset | tested (depends on fig5-dg-fos) |
| fig2a-pca-components, table3-pca-loadings, fig2a-two-trajectories, fig2b-pc-index-behaviours, fig3a-travel-development, fig3b-jumping-development, fig4-wheel-repetitive-jumping (Homecage Scan behaviour) | dataset | blocked: data missing |
| fig6-ihc-fos-confirmation (c-Fos IHC on sections) | dataset | blocked: data missing |
| fig7a-adult-induction-adcyap1, fig7b-postnatal-induction-adcyap1, fig7-cprg-expression (qRT-PCR) | dataset | blocked: data missing |
| fig8-induced-ko-no-hyperlocomotion, fig10a-baf53b-no-hyperlocomotion, fig11-placental-pacap-no-effect (behaviour) | dataset | blocked: data missing |
| fig9-postnatal-ko-pvn-fos, fig10c-baf53b-ish-knockdown (RNAscope, HCR) | dataset | blocked: data missing |
| fig10b-baf53b-restraint-weight-loss (body weight) | dataset | blocked: data missing |
| fig12b-pacapko-fasting-hypothermia, fig12b-pacapko-higher-tb, fig12d-pacapko-more-active, fig13a-sim1-tb-unchanged, fig13b-sim1-activity-unchanged, fig13c-pac1ko-tb-unchanged, fig13d-pac1ko-activity (telemetry) | dataset | blocked: data missing |

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import fos

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
print("dandiset", fos.DANDISET_ID, "version", fos.DANDISET_VERSION)

## Data check

What the dataset holds, by subject, hemisphere and genotype, and the checks of its description
against the files.

In [ ]:
A = fos.assets()
print("assets:", len(A), " total size (GB): %.1f" % (sum(a["size"] for a in A.values()) / 1e9))
# The other published version: same paths and blobs?
B, url = {}, f"{fos.API}/dandisets/{fos.DANDISET_ID}/versions/0.260701.1900/assets/?page_size=1000"
while url:
    r = requests.get(url, timeout=60).json(); B.update({x["path"]: x["blob"] for x in r["results"]}); url = r["next"]
print("version 0.260701.1900: assets", len(B), "| same paths and blobs as the pinned version:",
      B == {p: a["blob"] for p, a in A.items()})

meta = requests.get(f"{fos.API}/dandisets/{fos.DANDISET_ID}/versions/{fos.DANDISET_VERSION}/", timeout=60).json()
print("archive summary: subjects", meta["assetsSummary"]["numberOfSubjects"],
      "| samples", meta["assetsSummary"]["numberOfSamples"],
      "| species", meta["assetsSummary"]["species"])
part = fos.read_tsv("participants.tsv")
print("participants.tsv rows:", len(part), "| strains:", part.strain.value_counts().to_dict())
S = fos.samples()
print(S.to_string())
print("\nhemispheres by genotype:", S.genotype.value_counts().to_dict(),
      "| mice by genotype:", S.groupby("genotype").subject.nunique().to_dict())
print("mice with one hemisphere only:", S.groupby("subject").size().loc[lambda x: x == 1].index.tolist())
print("genotype in the subject id agrees with samples.tsv:",
      all((r.subject.endswith("ko") and r.genotype == "ko") or (r.subject.endswith("flox") and r.genotype == "flox")
          for r in S.itertuples()))
print("hemispheres with count tables:", int(S.count_tables.sum()), "| with masked atlas labels:", int(S.atlas_masked.sum()))

In [ ]:
import json
spim = json.loads(fos.read_text("SPIM.json"))
print("SPIM.json:", spim)
for p in [fos.label_path("45424flox", "Left"), fos.seg_paths("45424flox", "Left")[8000],
          f"derivatives/HeatmapsAtlasSpace/sub-45424flox/micr/sub-45424flox_sample-LeftHemisphere_acq-heatmap_res-25um_SPIM.ome.btf"]:
    tf = fos.open_tiff(p)
    print(p.split("/")[1], "shape", tf.series[0].shape, tf.series[0].dtype)
shapes = {}
for r in S.itertuples():
    shapes[(r.subject, r.hemisphere)] = fos.open_tiff(fos.label_path(r.subject, r.hemisphere)).series[0].shape
print("label image shapes (z, y, x):", pd.Series(shapes).value_counts().to_dict())
VOX_MM3 = np.prod(spim["PixelSize"]) * 1e-9   # 3.7 x 3.7 x 5 um voxels, in cubic mm
print("voxel volume (mm^3): %.3e" % VOX_MM3)

R = fos.regions()
print("atlas labels:", len(R), "| tracts and nerves:", int(R.tract_or_nerve.sum()), "| ventricles:", int(R.ventricle.sum()),
      "| cerebellum:", int(R.cerebellum.sum()), "| layer 1:", int(R.layer1.sum()), "| SPVI:", int(R.spvi.sum()),
      "| root and grey:", int(R.catch_all.sum()))
print("left after tracts, nerves and ventricles:", int((~(R.tract_or_nerve | R.ventricle)).sum()), "(paper 462)",
      "| included:", int(R.included.sum()), "(paper 431)")
inc = R[R.included].reset_index(drop=True)
print("\nincluded regions (%d):" % len(inc), " ".join(inc.acronym))
ex = R[~R.included & ~(R.tract_or_nerve | R.ventricle)]
print("\nexcluded besides tracts, nerves and ventricles (%d):" % len(ex))
for g in ("cerebellum", "layer1", "spvi", "catch_all"):
    print(" ", g, "(%d):" % int(ex[g].sum()), " ".join(ex[ex[g]].acronym))
print("\nlabels with the tract colour #cccccc (%d):" % int(R.tract_or_nerve.sum()), " ".join(R[R.tract_or_nerve].acronym))
print("labels with the ventricle colour #aaaaaa (%d):" % int(R.ventricle.sum()), " ".join(R[R.ventricle].acronym))

# The README's description of the stacks, and the voxel size the heatmap files state
readme = fos.read_text("README.md").decode()
print("README:", " | ".join(l.strip() for l in readme.splitlines() if "25 micron" in l or "CATNIP" in l or "publication" in l)[:900])
hm = fos.open_tiff("derivatives/HeatmapsAtlasSpace/sub-45424flox/micr/sub-45424flox_sample-LeftHemisphere_acq-heatmap_res-25um_SPIM.ome.btf")
import re as _re
print("heatmap OME PhysicalSizeX/Y/Z:", _re.findall(r'PhysicalSize[XYZ]="([0-9.]+)"', hm.ome_metadata),
      "| sidecar res-25um_SPIM.json:", json.loads(fos.read_text("derivatives/HeatmapsAtlasSpace/res-25um_SPIM.json"))["PixelSize"])
print("dataset_description GeneratedBy CodeURL:", json.loads(fos.read_text("dataset_description.json"))["GeneratedBy"][0]["CodeURL"])

# Recordings deposited more than once: identical blobs among the stacks and segmentations
blobs = pd.Series({p: a["blob"] for p, a in A.items() if p.endswith(".btf")})
print("btf assets:", len(blobs), "| sharing a blob with another asset:", int(blobs.duplicated(keep=False).sum()))

In [ ]:
# Artifact masks. The Supplemental Methods ("Statistical analysis") ignore counts of regions
# overlapping an artifact mask. The dataset holds no mask; it holds corrected atlas-space heatmaps
# (HeatmapsAtlasSpaceCorrected) for two hemispheres. Compare them with the uncorrected heatmaps.
for p in [p for p in A if p.startswith("derivatives/HeatmapsAtlasSpaceCorrected/") and p.endswith(".btf")]:
    a = fos.open_tiff(p).asarray()
    b = fos.open_tiff(p.replace("HeatmapsAtlasSpaceCorrected", "HeatmapsAtlasSpace")).asarray()
    d = a.astype(float) - b
    nzd = np.argwhere(d != 0)
    print(p.split("/")[2], "shape", a.shape, "| voxels > 0: uncorrected %d, corrected %d" % ((b > 0).sum(), (a > 0).sum()),
          "| voxels changed %d, of which set to 0: %d, raised: %d" % ((d != 0).sum(), ((a == 0) & (b > 0)).sum(), (d > 0).sum()),
          "| heatmap sum removed: %.2f%%" % (100 * (1 - a.sum() / b.sum())),
          "| changed voxels span z %d-%d, y %d-%d, x %d-%d" % (nzd[:, 0].min(), nzd[:, 0].max(), nzd[:, 1].min(), nzd[:, 1].max(), nzd[:, 2].min(), nzd[:, 2].max()))
print("atlas-space label images in the dataset:", [p for p in A if "space-" in p and "orig" not in p and "dseg" in p])


In [ ]:
# Check of the slab-wise count on a synthetic volume against scipy.ndimage.label (26-connectivity,
# each component assigned to the label holding most of its voxels, ties to the lowest label)
from scipy import ndimage
rng = np.random.default_rng(1362)
nz_, ny_, nx_ = 300, 60, 70
seg_ = np.zeros((nz_, ny_, nx_), bool)
for _ in range(1500):
    z, y, x = rng.integers(0, nz_), rng.integers(0, ny_), rng.integers(0, nx_)
    seg_[z:z + rng.integers(1, 6), y:y + rng.integers(1, 6), x:x + rng.integers(1, 6)] = True
for _ in range(5):                                   # a few components taller than the overlap
    y, x, z = rng.integers(0, ny_ - 2), rng.integers(0, nx_ - 2), rng.integers(0, 200)
    seg_[z:z + rng.integers(60, 90), y:y + 2, x:x + 2] = True
lab_ = np.zeros(seg_.shape, np.uint16)
lab_[:, 5:55, 5:65] = 1
for k in range(2, 9):
    z, y, x = rng.integers(0, nz_), rng.integers(0, ny_), rng.integers(0, nx_)
    lab_[max(0, z - 60):z + 60, max(5, y - 15):min(55, y + 15), max(5, x - 20):min(65, x + 20)] = k
ref, n_ref = ndimage.label(seg_, structure=np.ones((3, 3, 3)))
ref_all, ref_sized = np.zeros(fos.NLAB, int), np.zeros(fos.NLAB, int)        # components of at most OVERLAP planes
tall_all, tall_sized = np.zeros(fos.NLAB, int), np.zeros(fos.NLAB, int)      # taller ones
for i, sl in enumerate(ndimage.find_objects(ref), 1):
    m = ref[sl] == i
    l = np.bincount(lab_[sl][m], minlength=fos.NLAB).argmax()
    tall = sl[0].stop - sl[0].start > fos.OVERLAP
    (tall_all if tall else ref_all)[l] += 1
    (tall_sized if tall else ref_sized)[l] += fos.VMIN <= m.sum() <= fos.VMAX
syn = fos.count_slabs([lambda k: lab_[k.start:k.stop]], {1: lambda k: seg_[k.start:k.stop].astype(np.uint8)}, nz_)
ex_all, ex_sized = syn["all"][0][1] - ref_all, syn["sized"][0][1] - ref_sized
print("synthetic volume: %d components, %d taller than %d planes; slab count: %d components counted, %d pieces dropped"
      % (n_ref, tall_all.sum(), fos.OVERLAP, syn["all"][0][1].sum(), syn["dropped"][1]))
print("every component of at most %d planes counted, with the reference's label (counts minus reference are >= 0"
      " and no more than the tall components of that label): all components %s, 9 to 900 voxels %s"
      % (fos.OVERLAP, bool(((ex_all >= 0) & (ex_all <= tall_all)).all()), bool(((ex_sized >= 0) & (ex_sized <= tall_sized)).all())))
print("tall components measured whole (they start early enough in a slab): %d of %d; region volumes equal: %s"
      % (ex_all.sum(), tall_all.sum(), (syn["volume"][0] == np.bincount(lab_.ravel(), minlength=fos.NLAB)).all()))


### Recount of cells from the segmentations

The dataset's count tables cover only 13 of the 22 hemispheres, so the cells are counted here from
the binary FRST segmentations, which exist for all 22, at the five thresholds they all share
(4000 to 12000), and, for the 4 hemispheres segmented on the finer series 2000 to 12000, at its six
other thresholds as well. Each hemisphere is counted on the masked labels where the dataset has
them and, in the same pass, on the unmasked `AtlasLabel` too. The counting follows the Supplemental
Methods ("Cell counting"); see `fos.py` for where each parameter comes from. This is the long cell
(about 50 minutes, two processes). Thresholds 14000 to 26000, which 18 hemispheres hold, are
counted in the notebook `reproduce_bakalar_2023_thresholds_14000_26000.ipynb`, as counting them
here would take this notebook over its time limit; 28000 to 40000 were not run, for time.


In [ ]:
t0 = time.time()
hemis = list(zip(S.subject, S.hemisphere))
FINE = {(r.subject, r.hemisphere): tuple(t for t in r.seg_thresholds if t not in fos.THRESHOLDS)
        for r in S.itertuples() if max(r.seg_thresholds) <= 12000}
print("hemispheres on the finer series:", {f"{s} {h}": t for (s, h), t in FINE.items()})
RC = fos.count_all(hemis, {x: fos.THRESHOLDS + FINE.get(x, ()) for x in hemis}, workers=2, unmasked=True)
print("recount took %.1f min" % ((time.time() - t0) / 60))
rows = []
for (s, h), r in RC.items():
    for t in r["n_components"]:
        n_all, n_sized, n_small, n_big = r["n_components"][t]
        rows.append(dict(subject=s, hemisphere=h, threshold=t, components=n_all, within_9_900=n_sized,
                         below_9=n_small, above_900=n_big, too_tall_pieces_dropped=r["dropped"][t]))
comp = pd.DataFrame(rows)
print(comp.groupby("threshold")[["components", "within_9_900", "below_9", "above_900", "too_tall_pieces_dropped"]].agg(["sum", "count"]))
print("pieces of components too tall to measure (more than %d planes), as a fraction of all components: %.2e"
      % (fos.OVERLAP, comp.too_tall_pieces_dropped.sum() / comp.components.sum()))


In [ ]:
# Check the recount against the dataset's count tables (13 hemispheres)
rows = []
for r in S[S.count_tables].itertuples():
    rc = RC[(r.subject, r.hemisphere)]
    tabs = fos.count_table_paths(r.subject, r.hemisphere)
    for t in fos.THRESHOLDS:
        tab = fos.read_tsv(tabs[t])
        mine_all = rc["all"][t][tab.label_id.values]
        mine_sized = rc["sized"][t][tab.label_id.values]
        rows.append(dict(subject=r.subject, hemisphere=r.hemisphere, threshold=t, table_rows=len(tab),
                         table_total=int(tab.cell_count.sum()), recount_all=int(mine_all.sum()),
                         recount_9_900=int(mine_sized.sum()),
                         abs_diff_all_pct=100 * np.abs(mine_all - tab.cell_count.values).sum() / tab.cell_count.sum(),
                         r_all=np.corrcoef(mine_all, tab.cell_count)[0, 1],
                         volume_equal=bool((rc["volume"][tab.label_id.values] == tab.label_volume_pixels.values).all()),
                         masked_labels=r.atlas_masked))
check = pd.DataFrame(rows)
print(check.round(4).to_string())
print("\nsummed absolute per-region difference, all components against the tables: median %.2f%%, max %.2f%%"
      % (check.abs_diff_all_pct.median(), check.abs_diff_all_pct.max()))
print("components of 9 to 900 voxels as a fraction of the table totals: %.2f to %.2f"
      % ((check.recount_9_900 / check.table_total).min(), (check.recount_9_900 / check.table_total).max()))
print("region volumes equal to the tables' label_volume_pixels in every row:", check.volume_equal.all())

In [ ]:
# The same check on the regions the claims name, at threshold 8000
T4 = ["DG-sg", "DG-po", "LGv", "IGL", "SNr", "AUDp2/3", "DTN", "IO", "AUDpo2/3", "LGd", "PST", "LRN", "STN", "PGRNl", "NB"]
ids = R.set_index("acronym").loc[T4, "label_id"].values
rows = []
for r in S[S.count_tables].itertuples():
    tab = fos.read_tsv(fos.count_table_paths(r.subject, r.hemisphere)[8000]).set_index("label_id")
    rc = RC[(r.subject, r.hemisphere)]
    rows.append(pd.Series(rc["all"][8000][ids] - tab.loc[ids, "cell_count"].values, index=T4, name=f"{r.subject} {r.hemisphere[0]}"))
print("recount (all components) minus table count, threshold 8000:")
print(pd.DataFrame(rows).to_string())

# Duplicate hemispheres: identical recount vectors
V = pd.DataFrame({f"{s} {h[0]}": RC[(s, h)]["all"][8000][inc.label_id.values] for s, h in hemis})
cor = V.corr().values.copy()
np.fill_diagonal(cor, np.nan)
print("\nhighest correlation of per-region counts between two hemispheres: %.4f; identical pairs: %d"
      % (np.nanmax(cor), int(sum((V.iloc[:, i] == V.iloc[:, j]).all() for i in range(V.shape[1]) for j in range(i + 1, V.shape[1])))))

Clock and conditions: the data are images with no time series or trial table, so there is no clock
to check. The genotype of each hemisphere comes from `samples.tsv` and agrees with the subject ids.
The paper reports 8 PACAP fl/fl hemispheres from 6 mice and 11 PACAPko hemispheres from 6 mice; the
dataset holds 10 and 12 hemispheres, from 6 and 7 mice, and does not say which were used.

## Readings

Every claim below is tested on all 22 hemispheres (claims file, reading 4b; the dataset marks no
hemisphere as used) under these readings, named in the claim criteria:

- (1) FRST threshold: 4000, 6000, 8000, 10000, 12000 (not marked in the dataset; these five are the
  ones all 22 segmentations share). The other thresholds the derivatives hold are run as well:
  14000 to 26000, which 18 hemispheres hold, in the threshold notebook (28000 to 40000 were not
  run, for time), and the finer series
  2000 to 12000 of the other 4 hemispheres here (variant "own series" below).
- (2) Region size: (a) the region's voxel count in that hemisphere's label image (`density`, cells
  per cubic mm), (b) the atlas region volume, which is one constant per region, so that dividing by
  it changes neither U, p nor the percentage: (b) is run as the raw counts (`raw`).
- (3) IQR outlier rule: (a) within each genotype, (b) both genotypes pooled.
- (5) Percentage: ratio of means and ratio of medians after outlier removal.
- (6) BKY alpha: 0.05, with 0.1 shown.

Counting rule: the paper's 9 to 900 voxel rule is the main analysis; counting every component, the
rule that the dataset's own count tables follow, is shown as a variant. Two more variants: the 13
hemispheres the dataset's count tables cover, analysed from those tables at every threshold they
hold; and every choice of 8 PACAP fl/fl hemispheres from 6 mice and 11 PACAPko hemispheres from 6
mice, the paper's numbers (step 1.6 of the protocol; the paper does not say which were left out).
Four more variants of the main rule, each under the 20 readings: counting on the unmasked
`AtlasLabel` for all 22 hemispheres (6 hemispheres have no masked label image, so the main analysis
mixes the two); the 18 hemispheres that hold thresholds 14000 to 40000; leaving out the 2
hemispheres with a corrected heatmap, the most an artifact exclusion could remove for them (the
dataset holds no artifact mask, and its corrected heatmaps are in atlas space, for which it holds no
label image); and the 4 hemispheres of the finer series at each threshold of their own series.

In [ ]:
def matrix(hemis, t, rule="sized", size="density", labels=""):
    # Rows hemispheres, columns included regions; NaN where the region has no voxels in that hemisphere.
    # t is one threshold or a dict hemisphere -> threshold; labels "" (masked where the dataset has them) or "_u" (unmasked)
    rows = []
    for s, h in hemis:
        r = RC[(s, h)]
        c = r[rule + labels][t[(s, h)] if isinstance(t, dict) else t][inc.label_id.values].astype(float)
        v = r["volume" + labels][inc.label_id.values].astype(float)
        rows.append(np.where(v > 0, c / (v * VOX_MM3) if size == "density" else c, np.nan))
    m = pd.DataFrame(rows, columns=inc.acronym.values)
    return m.loc[:, m.notna().any()]

geno = S.genotype.reset_index(drop=True)
GRID = [(t, size, iqr) for t in fos.THRESHOLDS for size in ("density", "raw") for iqr in ("within", "pooled")]
TESTS = {}
for rule in ("sized", "all"):
    for t, size, iqr in GRID:
        tab = fos.region_tests(matrix(hemis, t, rule, size), geno, iqr)
        tab["p_adj_a10"] = multipletests(tab.p.values, alpha=0.1, method="fdr_tsbky")[1]
        TESTS[(rule, t, size, iqr)] = tab
print("analyses run:", len(TESTS), "| regions tested per analysis:", sorted({len(v) for v in TESTS.values()}))

In [ ]:
# Variants of the main counting rule (9 to 900 voxels), each under the 20 readings of threshold, size and IQR rule:
# - unmasked: all 22 hemispheres counted on the unmasked AtlasLabel, the one label image every hemisphere has;
# - 18 hemispheres: without the 4 on the finer series, the set that thresholds 14000 to 40000 are run on;
# - no corrected heatmap: without the 2 hemispheres that have a corrected heatmap (45670ko Left, 45925flox Left),
#   the most the unknown artifact exclusion could remove for them;
# - own series: the 18 hemispheres at each shared threshold and the 4 of the finer series at each threshold of
#   their own series (2000 to 12000), the same for all 4 (thresholds chosen per hemisphere, Supplemental Methods,
#   "Segmentation": "An expert user determined which threshold(s) to use").
def subset_tests(hs, t, size, iqr, labels=""):
    g = pd.Series([geno[hemis.index(x)] for x in hs])
    return fos.region_tests(matrix(hs, t, "sized", size, labels), g, iqr)

h18 = [x for x in hemis if x not in FINE]
hnc = [x for x in hemis if x not in [("45670ko", "Left"), ("45925flox", "Left")]]
fine_series = sorted(set.intersection(*[set(fos.THRESHOLDS + v) for v in FINE.values()]))
VAR = {}
for t, size, iqr in GRID:
    VAR[("unmasked", t, size, iqr)] = subset_tests(hemis, t, size, iqr, "_u")
    VAR[("18 hemispheres", t, size, iqr)] = subset_tests(h18, t, size, iqr)
    VAR[("no corrected heatmap", t, size, iqr)] = subset_tests(hnc, t, size, iqr)
OWN = {}
for t in fos.THRESHOLDS:
    for tf in fine_series:
        for size in ("density", "raw"):
            for iqr in ("within", "pooled"):
                OWN[(t, tf, size, iqr)] = subset_tests(hemis, {x: (tf if x in FINE else t) for x in hemis}, size, iqr)
print("hemispheres: 18 hemispheres", geno[[hemis.index(x) for x in h18]].value_counts().to_dict(),
      "| no corrected heatmap", geno[[hemis.index(x) for x in hnc]].value_counts().to_dict())
print("finer-series thresholds used for the 4:", fine_series, "| own-series analyses:", len(OWN))


## fig5-dg-fos

> In whole-brain c-Fos counts after the early dark phase, PACAPko hemispheres have a higher density
> of c-Fos-positive cells than PACAP fl/fl in the dentate gyrus granule cell layer and polymorph
> layer, the only two regions significant after correction for multiple testing; PACAPko densities
> are 224% and 326% of control.

Criterion (claims file): reproduced when, under the readings, DG-sg and DG-po both have PACAPko
higher with BKY-adjusted p < 0.05; partially reproduced when one of the two holds; if the readings
of (1) to (4) give different outcomes, blocked as `methods_underspecified` with every result shown.
Expected values (reported, not deciding alone): raw p 2.65e-05 and 1.85e-04, U 88 and 85, PACAPko
at 224.23% and 325.82% of PACAP fl/fl.

In [ ]:
def dg_outcome(tab):
    ok = [(r in tab.index) and tab.loc[r, "p_adj"] < 0.05 and tab.loc[r, "mean_ko"] > tab.loc[r, "mean_flox"] for r in ("DG-sg", "DG-po")]
    return "reproduced" if all(ok) else ("partial" if any(ok) else "not")

def dg_table(rule):
    rows = []
    for t, size, iqr in GRID:
        tab = TESTS[(rule, t, size, iqr)]
        d = dict(threshold=t, size=size, iqr=iqr)
        for r in ("DG-sg", "DG-po"):
            x = tab.loc[r]
            d.update({f"{r} n": f"{int(x.n_ko)}/{int(x.n_flox)}", f"{r} U": x.U, f"{r} p": x.p, f"{r} p_adj": x.p_adj,
                      f"{r} p_adj a.1": x.p_adj_a10, f"{r} %mean": x.pct_mean, f"{r} %median": x.pct_median})
        d["n regions p_adj<0.05"] = int((tab.p_adj < 0.05).sum())
        d["outcome"] = dg_outcome(tab)
        rows.append(d)
    return pd.DataFrame(rows)

dg_main = dg_table("sized")
print("Main analysis: components of 9 to 900 voxels, 22 hemispheres (n is PACAPko/PACAP fl/fl after outliers)")
print(dg_main.to_string(float_format=lambda v: "%.4g" % v))
print("\noutcomes over the 20 readings:", dg_main.outcome.value_counts().to_dict())
print("outcome by size and IQR reading (over the five thresholds):")
print(dg_main.groupby(["size", "iqr"]).outcome.value_counts().unstack(fill_value=0))

In [ ]:
dg_all = dg_table("all")
print("Variant: every component counted (the dataset's count-table rule), 22 hemispheres")
print(dg_all.to_string(float_format=lambda v: "%.4g" % v))
print("\noutcomes:", dg_all.outcome.value_counts().to_dict())

In [ ]:
# Which regions are significant after BKY, main analysis, density and within-genotype IQR
for t in fos.THRESHOLDS:
    tab = TESTS[("sized", t, "density", "within")]
    s = tab[tab.p_adj < 0.05]
    print(t, len(s), "regions:", ", ".join(f"{r} ({'up' if x.mean_ko > x.mean_flox else 'down'})" for r, x in s.iterrows()))

In [ ]:
# Per hemisphere DG densities (cells per cubic mm, 9 to 900 voxel components), by animal
rows = []
for t in fos.THRESHOLDS:
    m = matrix(hemis, t, "sized", "density")
    for i, (s, h) in enumerate(hemis):
        rows.append(dict(threshold=t, subject=s, hemisphere=h, genotype=geno[i], DG_sg=m.loc[i, "DG-sg"], DG_po=m.loc[i, "DG-po"]))
per = pd.DataFrame(rows)
print(per.pivot_table(index=["genotype", "subject", "hemisphere"], columns="threshold", values=["DG_sg", "DG_po"]).round(0).to_string())
print("\nrange of plotted DG-sg density: %.0f to %.0f; DG-po: %.0f to %.0f cells per cubic mm"
      % (per.DG_sg.min(), per.DG_sg.max(), per.DG_po.min(), per.DG_po.max()))
# By animal: mean over the hemispheres of each mouse, then PACAPko above every PACAP fl/fl mouse?
an = per.groupby(["threshold", "genotype", "subject"])[["DG_sg", "DG_po"]].mean().reset_index()
for t in fos.THRESHOLDS:
    a = an[an.threshold == t]
    for r in ("DG_sg", "DG_po"):
        ko, fl = a[a.genotype == "ko"][r], a[a.genotype == "flox"][r]
        print(t, r, "mouse means: PACAPko %.0f to %.0f, PACAP fl/fl %.0f to %.0f; PACAPko mice above the highest fl/fl mouse: %d of %d"
              % (ko.min(), ko.max(), fl.min(), fl.max(), (ko > fl.max()).sum(), len(ko)))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for k, r in enumerate(("DG_sg", "DG_po")):
    for j, t in enumerate(fos.THRESHOLDS):
        for g, c, dx in (("flox", "tab:orange", -0.15), ("ko", "tab:blue", 0.15)):
            y = per[(per.threshold == t) & (per.genotype == g)][r]
            ax[k].scatter(np.full(len(y), j + dx), y, s=12, c=c, label=("PACAP fl/fl" if g == "flox" else "PACAPko") if j == 0 else None)
    ax[k].set_xticks(range(5), fos.THRESHOLDS); ax[k].set_xlabel("FRST threshold"); ax[k].set_title(r.replace("_", "-"))
ax[0].set_ylabel("cells per cubic mm"); ax[0].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Variant: the 13 hemispheres the dataset's count tables cover (one per mouse), from those tables,
# every threshold they hold; density = cell_count / label_volume_pixels; IQR within genotype and pooled
TAB = {}
for r in S[S.count_tables].itertuples():
    for t, p in fos.count_table_paths(r.subject, r.hemisphere).items():
        TAB[(r.subject, r.hemisphere, t)] = fos.read_tsv(p).set_index("label_id")
tab_hemis = [(r.subject, r.hemisphere) for r in S[S.count_tables].itertuples()]
tab_geno = S[S.count_tables].genotype.reset_index(drop=True)
common = sorted(set.intersection(*[{t for (s, h, t) in TAB if (s, h) == x} for x in tab_hemis]))
print("count-table hemispheres:", len(tab_hemis), tab_geno.value_counts().to_dict(), "| thresholds all of them hold:", common)
rows = []
for t in common:
    for size in ("density", "raw"):
        m = []
        for s, h in tab_hemis:
            x = TAB[(s, h, t)].loc[inc.label_id.values]
            v = x.label_volume_pixels.values.astype(float)
            m.append(np.where(v > 0, x.cell_count.values / (v * VOX_MM3) if size == "density" else x.cell_count.values, np.nan))
        m = pd.DataFrame(m, columns=inc.acronym.values); m = m.loc[:, m.notna().any()]
        for iqr in ("within", "pooled"):
            tab = fos.region_tests(m, tab_geno, iqr)
            rows.append(dict(threshold=t, size=size, iqr=iqr, **{f"{r} {k}": tab.loc[r, k] for r in ("DG-sg", "DG-po") for k in ("U", "p", "p_adj", "pct_mean")},
                             n_sig=int((tab.p_adj < 0.05).sum()), n_raw01=int((tab.p <= 0.01).sum()), outcome=dg_outcome(tab)))
tabvar = pd.DataFrame(rows)
print(tabvar.to_string(float_format=lambda v: "%.4g" % v))
print("\noutcomes:", tabvar.outcome.value_counts().to_dict())

In [ ]:
# Variant (protocol step 1.6): every choice of 8 PACAP fl/fl hemispheres from 6 mice and 11 PACAPko
# hemispheres from 6 mice, as the paper reports; main counting rule, at each threshold, each size and IQR reading
import itertools
fl_h = [x for x, g in zip(hemis, geno) if g == "flox"]
ko_h = [x for x, g in zip(hemis, geno) if g == "ko"]
two = [s for s in sorted({s for s, _ in fl_h}) if sum(1 for x in fl_h if x[0] == s) == 2]
fl_sets = [[x for x in fl_h if x not in ((a, ha), (b, hb))]
           for a, b in itertools.combinations(two, 2) for ha in ("Left", "Right") for hb in ("Left", "Right")]
singles = [s for s in sorted({s for s, _ in ko_h}) if sum(1 for x in ko_h if x[0] == s) == 1]
ko_sets = [[x for x in ko_h if x[0] != s] for s in singles]
print("PACAP fl/fl choices:", len(fl_sets), "| PACAPko choices (leave out one of the one-hemisphere mice %s):" % singles, len(ko_sets))
FULL = {}
for t in fos.THRESHOLDS:
    for size in ("density", "raw"):
        FULL[(t, size)] = matrix(hemis, t, "sized", size)
key = {x: i for i, x in enumerate(hemis)}
SUB = []
for fi, fs in enumerate(fl_sets):
    for ki, ks in enumerate(ko_sets):
        idx = [key[x] for x in fs + ks]
        g = geno[idx].reset_index(drop=True)
        for (t, size), m in FULL.items():
            for iqr in ("within", "pooled"):
                tab = fos.region_tests(m.iloc[idx].reset_index(drop=True), g, iqr)
                SUB.append(dict(fl=fi, ko=ki, threshold=t, size=size, iqr=iqr, outcome=dg_outcome(tab),
                                n_sig=int((tab.p_adj < 0.05).sum()),
                                **{f"{r} {k}": tab.loc[r, k] for r in T4 for k in ("U", "p", "p_adj", "pct_mean", "pct_median")}))
SUB = pd.DataFrame(SUB)
print("subset analyses:", len(SUB))
print(SUB.groupby(["threshold", "size", "iqr"]).outcome.value_counts().unstack(fill_value=0).to_string())
print("\nover all subset analyses:", SUB.outcome.value_counts().to_dict())
print("DG-sg raw p range %.2e to %.2e, U %d to %d; DG-po raw p %.2e to %.2e, U %d to %d"
      % (SUB["DG-sg p"].min(), SUB["DG-sg p"].max(), SUB["DG-sg U"].min(), SUB["DG-sg U"].max(),
         SUB["DG-po p"].min(), SUB["DG-po p"].max(), SUB["DG-po U"].min(), SUB["DG-po U"].max()))
print("subset analyses where DG-sg has U = 88 (complete separation, as Table 4): %d of %d" % ((SUB["DG-sg U"] == 88).sum(), len(SUB)))

In [ ]:
# Masked and unmasked label images, for the hemispheres that have both: DG cells (9 to 900 voxels) and densities
rows = []
for s, h in [x for x in hemis if S.set_index(["subject", "hemisphere"]).loc[x, "atlas_masked"]]:
    r = RC[(s, h)]
    for t in fos.THRESHOLDS:
        for reg in ("DG-sg", "DG-po"):
            i = int(R.set_index("acronym").loc[reg, "label_id"])
            dm = r["sized"][t][i] / (r["volume"][i] * VOX_MM3); du = r["sized_u"][t][i] / (r["volume_u"][i] * VOX_MM3)
            rows.append(dict(subject=s, hemisphere=h, threshold=t, region=reg, cells_masked=r["sized"][t][i], cells_unmasked=r["sized_u"][t][i],
                             voxels_masked=r["volume"][i], voxels_unmasked=r["volume_u"][i], density_masked=dm, density_unmasked=du,
                             density_diff_pct=100 * (du - dm) / dm))
mu = pd.DataFrame(rows)
print("hemispheres with both label images:", mu.groupby(["subject", "hemisphere"]).ngroups)
print(mu[mu.threshold == 8000].round(1).to_string())
print("\ndensity difference unmasked - masked, % of masked, over hemispheres and the 5 thresholds:")
print(mu.groupby("region").density_diff_pct.describe().round(3).to_string())
allreg = []
for s, h in mu[["subject", "hemisphere"]].drop_duplicates().itertuples(index=False):
    r = RC[(s, h)]; v = r["volume"][inc.label_id.values]; vu = r["volume_u"][inc.label_id.values]
    allreg.append(dict(subject=s, hemisphere=h, regions_volume_differs=int((v != vu).sum()),
                       regions_count_differs_8000=int((r["sized"][8000][inc.label_id.values] != r["sized_u"][8000][inc.label_id.values]).sum()),
                       voxels_masked_pct_of_unmasked=100 * v.sum() / vu.sum()))
print(pd.DataFrame(allreg).round(2).to_string())


In [ ]:
# Outcomes of the variants (main counting rule) over their readings
def var_table(name):
    rows = []
    for t, size, iqr in GRID:
        tab = VAR[(name, t, size, iqr)]
        rows.append(dict(threshold=t, size=size, iqr=iqr, **{f"{r} {k}": tab.loc[r, k] for r in ("DG-sg", "DG-po") for k in ("U", "p", "p_adj", "pct_mean")},
                         n_sig=int((tab.p_adj < 0.05).sum()), outcome=dg_outcome(tab)))
    return pd.DataFrame(rows)
for name in ("unmasked", "18 hemispheres", "no corrected heatmap"):
    v = var_table(name)
    print(name); print(v.to_string(float_format=lambda x: "%.4g" % x))
    print("outcomes:", v.outcome.value_counts().to_dict()); print()
own = pd.DataFrame([dict(threshold=t, fine_threshold=tf, size=size, iqr=iqr, outcome=dg_outcome(tab),
                         **{f"{r} p_adj": tab.loc[r, "p_adj"] for r in ("DG-sg", "DG-po")}) for (t, tf, size, iqr), tab in OWN.items()])
print("own series: the 18 at each shared threshold, the 4 of the finer series at each of theirs; outcomes by shared threshold, size and IQR rule:")
print(own.groupby(["threshold", "size", "iqr"]).outcome.value_counts().unstack(fill_value=0).to_string())
print("outcomes over all %d:" % len(own), own.outcome.value_counts().to_dict())


**Result: blocked (methods_underspecified).** On all 22 hemispheres, with the paper's 9 to 900
voxel rule, the 20 readings (five thresholds, two region sizes, two outlier rules) give different
outcomes: DG-sg and DG-po are both higher in PACAPko with BKY-adjusted p < 0.05 in 12, one of the
two in 3 and neither in 5, so by the criterion the claim is blocked. With region density (the
region's voxels in each hemisphere) and outliers removed within each genotype, both hold at every
threshold (DG-sg adjusted p 0.0013 to 0.035, DG-po 0.0014 to 0.0088); with outliers removed from
both genotypes pooled, or with raw counts, they fail at some thresholds from 8000 up. PACAPko is
higher in both regions under every reading: DG-sg at 199.7% to 238.9% of PACAP fl/fl (ratio of
means; the paper 224.23%) and DG-po at 260.8% to 546.8% (the paper 325.82%). At threshold 4000 every
PACAPko mouse is above every PACAP fl/fl mouse in both regions (mouse means). DG-sg and DG-po are
not the only significant regions: under the density and within-genotype reading, 11 to 38 regions
have adjusted p < 0.05. Counting every component, as the dataset's count tables do, gives 15
readings with both, 2 with one and 3 with neither. On the 13 hemispheres of the count tables
neither region reaches adjusted p < 0.05 under any reading (raw p 0.0012 to 0.035). Over every
choice of 8 PACAP fl/fl and 11 PACAPko hemispheres (the paper's numbers), 480 of 960 subset
analyses give both regions, 128 one and 352 neither, and in 150 of them DG-sg reaches U = 88 with
raw p 2.65e-05, the values of Table 4.

The other variants do not settle the readings either. Counting on the unmasked label images for
all 22 hemispheres gives 14 readings with both, 3 with one and 3 with neither. The cell counts in
DG are the same on both label images, but the unmasked regions are larger, so DG densities are
0% to 17% lower (DG-sg) and 0% to 12% lower (DG-po). Leaving out the 2 hemispheres with a
corrected heatmap gives 7, 6 and 7. The 18 hemispheres that hold the higher thresholds give 5, 2
and 13 at 4000 to 12000. In the threshold notebook they give neither region in 55 of 56 analyses
at 14000 to 26000, and one region in the other. Letting the 4 hemispheres of the finer series take
each threshold of their own series gives 133 of 220 analyses with both, 37 with one and 50 with
neither.

## table4-lenient-regions

> At an uncorrected cutoff of raw p <= 0.01, 13 further regions differ in c-Fos cell density between
> PACAPko and PACAP fl/fl, including substantia nigra reticular part, subthalamic nucleus and
> intergeniculate leaflet.

Criterion: same pipeline, tests and readings as fig5-dg-fos. Reproduced when SNr, STN and IGL have
raw p <= 0.01 with PACAPko higher, and the number of other regions (besides DG-sg and DG-po) with
raw p <= 0.01 is within 13 +/- 1.3; partially reproduced when one or two of SNr, STN and IGL meet
the cutoff. The overlap with the 15 regions of Table 4 is reported.

In [ ]:
def lenient(tab):
    three = {r: bool(r in tab.index and tab.loc[r, "p"] <= 0.01 and tab.loc[r, "mean_ko"] > tab.loc[r, "mean_flox"]) for r in ("SNr", "STN", "IGL")}
    hits = tab[tab.p <= 0.01].index
    n_other = int(len([r for r in hits if r not in ("DG-sg", "DG-po")]))
    within = abs(n_other - 13) <= 1.3
    k = sum(three.values())
    out = "reproduced" if k == 3 and within else ("partial" if k > 0 else "not")
    return three, n_other, len(set(hits) & set(T4)), out

def lenient_table(rule):
    rows = []
    for t, size, iqr in GRID:
        three, n, ov, out = lenient(TESTS[(rule, t, size, iqr)])
        tab = TESTS[(rule, t, size, iqr)]
        rows.append(dict(threshold=t, size=size, iqr=iqr, **{f"{r} p": tab.loc[r, "p"] for r in ("SNr", "STN", "IGL")},
                         **{f"{r} ok": v for r, v in three.items()}, other_regions_p01=n, overlap_with_table4_of_15=ov, outcome=out))
    return pd.DataFrame(rows)

len_main = lenient_table("sized")
print("Main analysis, 22 hemispheres")
print(len_main.to_string(float_format=lambda v: "%.3g" % v))
print("\noutcomes:", len_main.outcome.value_counts().to_dict())
print("\nVariant, every component counted:")
len_all = lenient_table("all")
print(len_all[["threshold", "size", "iqr", "SNr ok", "STN ok", "IGL ok", "other_regions_p01", "overlap_with_table4_of_15", "outcome"]].to_string())
print("outcomes:", len_all.outcome.value_counts().to_dict())

In [ ]:
# Paper-sized subsets (8 and 11 hemispheres): SNr, STN and IGL at raw p <= 0.01, PACAPko higher
for r in ("SNr", "STN", "IGL"):
    SUB[f"{r} ok"] = (SUB[f"{r} p"] <= 0.01) & (SUB[f"{r} pct_mean"] > 100)
SUB["n_three"] = SUB[["SNr ok", "STN ok", "IGL ok"]].sum(axis=1)
print("subset analyses by how many of SNr, STN, IGL meet the cutoff:")
print(SUB.groupby(["threshold", "size", "iqr"]).n_three.value_counts().unstack(fill_value=0).to_string())

In [ ]:
# Variants of the main rule (unmasked labels, 18 hemispheres, no corrected-heatmap hemispheres, own series)
rows = []
for (name, t, size, iqr), tab in VAR.items():
    three, n, ov, out = lenient(tab)
    rows.append(dict(variant=name, threshold=t, size=size, iqr=iqr, **{f"{r} p": tab.loc[r, "p"] for r in ("SNr", "STN", "IGL")},
                     **{f"{r} ok": v for r, v in three.items()}, other_regions_p01=n, overlap_with_table4_of_15=ov, outcome=out))
lv = pd.DataFrame(rows)
print(lv.to_string(float_format=lambda v: "%.3g" % v))
print("\noutcomes by variant:"); print(lv.groupby("variant").outcome.value_counts().unstack(fill_value=0).to_string())
lo = pd.DataFrame([dict(threshold=t, fine_threshold=tf, size=size, iqr=iqr, **dict(zip(("three", "n_other", "overlap", "outcome"), lenient(tab))))
                   for (t, tf, size, iqr), tab in OWN.items()])
lo["n_three"] = lo.three.apply(lambda d: sum(d.values()))
print("\nown series: outcomes", lo.outcome.value_counts().to_dict(), "| how many of SNr, STN, IGL meet the cutoff:", lo.n_three.value_counts().sort_index().to_dict(),
      "| other regions at p <= 0.01: %d to %d" % (lo.n_other.min(), lo.n_other.max()))


**Result: blocked (depends on fig5-dg-fos, methods_underspecified).** Under every one of the 20
readings on 22 hemispheres the claim's own test gives a partial result: IGL meets raw p <= 0.01
with PACAPko higher in all 20, SNr in 16, STN only at threshold 4000 with region density (2 of 20),
and the number of other regions at raw p <= 0.01 is 9 to 44, against 13 +/- 1.3 (within tolerance
in 1 reading, raw counts pooled at 8000, where STN fails). The regions found overlap 4 to 13 of
the 15 of Table 4. Counting every component gives the same partial result in all 20 readings, with
STN meeting the cutoff in 7. The variants are partial in all 20 readings on unmasked labels and in
15 of 20 without the corrected-heatmap hemispheres (5 not reproduced). On the 18 hemispheres they
are partial in 14 of 20 at 4000 to 12000, and in the threshold notebook they are not reproduced in
54 of 56 analyses at 14000 to 26000, where SNr and STN never meet the cutoff. With the finer
series, 191 of 220 are partial and 29 not reproduced; all three of SNr, STN and IGL meet the cutoff
in 33 of them, but never with the count of other regions within tolerance. So the claim's own test
depends on the threshold as well.

## table4-region-values

> Table 4 gives, for each of the 15 regions with raw p <= 0.01, the Mann-Whitney U and PACAPko density
> as a percentage of PACAP fl/fl; every region but DTN (45%) is higher in PACAPko (167% to 326%).

Criterion: same pipeline, tests and readings as fig5-dg-fos; U is that of the PACAPko sample.
Reproduced when, under one reading, the direction (PACAPko above or below 100%) matches for all 13
regions and at least 10 of the 13 have both U and percentage within tolerance; partially reproduced
when the direction matches for all and fewer are within tolerance; not reproduced when any direction
differs.

In [ ]:
EXP = {"LGv": (73, 170.42), "IGL": (72, 258.62), "SNr": (80, 275.91), "AUDp2/3": (73, 261.06), "DTN": (9, 45.21),
       "IO": (76, 297.49), "AUDpo2/3": (76, 198.26), "LGd": (76, 276.46), "PST": (68, 208.73), "LRN": (76, 293.57),
       "STN": (70, 196.47), "PGRNl": (75, 239.16), "NB": (75, 167.26)}
TOL = {"LGv": (7.3, 17.0), "IGL": (7.2, 25.9), "SNr": (8.0, 27.6), "AUDp2/3": (7.3, 26.1), "DTN": (0.9, 4.5),
       "IO": (7.6, 29.7), "AUDpo2/3": (7.6, 19.8), "LGd": (7.6, 27.6), "PST": (6.8, 20.9), "LRN": (7.6, 29.4),
       "STN": (7.0, 19.6), "PGRNl": (7.5, 23.9), "NB": (7.5, 16.7)}

def values_eval(get, pct):
    # get(region, key) -> value; pct is "pct_mean" or "pct_median"
    dirs = [(get(r, pct) > 100) == (EXP[r][1] > 100) for r in EXP]
    within = [abs(get(r, "U") - EXP[r][0]) <= TOL[r][0] and abs(get(r, pct) - EXP[r][1]) <= TOL[r][1] for r in EXP]
    out = ("reproduced" if all(dirs) and sum(within) >= 10 else "partial") if all(dirs) else "not"
    return sum(dirs), sum(within), out

rows = []
for rule in ("sized", "all"):
    for t, size, iqr in GRID:
        tab = TESTS[(rule, t, size, iqr)]
        for pct in ("pct_mean", "pct_median"):
            nd, nw, out = values_eval(lambda r, k: tab.loc[r, k], pct)
            rows.append(dict(rule=rule, threshold=t, size=size, iqr=iqr, pct=pct, directions_match=nd, within_tol=nw, outcome=out))
tv = pd.DataFrame(rows)
print(tv.to_string())
print("\nmain rule outcomes:", tv[tv.rule == "sized"].outcome.value_counts().to_dict(),
      "| every-component rule:", tv[tv.rule == "all"].outcome.value_counts().to_dict())

In [ ]:
# The values themselves, main rule, density, IQR within genotype, at each threshold (U of PACAPko; % by ratio of means)
for t in fos.THRESHOLDS:
    tab = TESTS[("sized", t, "density", "within")]
    d = pd.DataFrame({"U": tab.loc[list(EXP), "U"], "U paper": [EXP[r][0] for r in EXP],
                      "% mean": tab.loc[list(EXP), "pct_mean"], "% median": tab.loc[list(EXP), "pct_median"],
                      "% paper": [EXP[r][1] for r in EXP], "raw p": tab.loc[list(EXP), "p"],
                      "n ko/flox": [f"{int(tab.loc[r, 'n_ko'])}/{int(tab.loc[r, 'n_flox'])}" for r in EXP]})
    print("threshold", t); print(d.to_string(float_format=lambda v: "%.4g" % v)); print()

In [ ]:
# Paper-sized subsets: how many of the 13 regions match in direction and are within tolerance
rows = []
for _, x in SUB.iterrows():
    for pct in ("pct_mean", "pct_median"):
        nd, nw, out = values_eval(lambda r, k: x[f"{r} {k}"], pct)
        rows.append(dict(threshold=x.threshold, size=x["size"], iqr=x.iqr, pct=pct, directions=nd, within=nw, outcome=out))
sv = pd.DataFrame(rows)
print(sv.groupby(["threshold", "size", "iqr", "pct"]).outcome.value_counts().unstack(fill_value=0).to_string())
print("\nover all:", sv.outcome.value_counts().to_dict(), "| most regions within tolerance in any subset analysis:", sv.within.max())

In [ ]:
# Variants of the main rule (unmasked labels, 18 hemispheres, no corrected-heatmap hemispheres, own series)
rows = []
for (name, t, size, iqr), tab in VAR.items():
    for pct in ("pct_mean", "pct_median"):
        nd, nw, out = values_eval(lambda r, k: tab.loc[r, k], pct)
        rows.append(dict(variant=name, threshold=t, size=size, iqr=iqr, pct=pct, directions_match=nd, within_tol=nw, outcome=out))
for (t, tf, size, iqr), tab in OWN.items():
    for pct in ("pct_mean", "pct_median"):
        nd, nw, out = values_eval(lambda r, k: tab.loc[r, k], pct)
        rows.append(dict(variant="own series %d" % tf, threshold=t, size=size, iqr=iqr, pct=pct, directions_match=nd, within_tol=nw, outcome=out))
vv = pd.DataFrame(rows)
vv["group"] = vv.variant.str.replace(r" \d+$", "", regex=True)
print(vv.groupby("group").agg(analyses=("outcome", "size"), directions_min=("directions_match", "min"),
                              within_max=("within_tol", "max")).to_string())
print(vv.groupby("group").outcome.value_counts().unstack(fill_value=0).to_string())


**Result: blocked (depends on fig5-dg-fos, methods_underspecified).** Under every reading
(both counting rules, five thresholds, two region sizes, two outlier rules, both percentages), the
direction of all 13 regions matches Table 4 (DTN lower in PACAPko, the rest higher), but at most 1
of the 13 has both U and percentage within tolerance, against the 10 the criterion asks: the claim's
own test gives partially reproduced. The U values are higher than the paper's (for example IGL 100
to 116 against 72) because 22 hemispheres give a larger maximum U (up to 12 x 10 = 120 against
88). On the paper-sized subsets, at most 8 of 13 are within tolerance, and 24 of 1920 subset
analyses have a direction that differs. The unmasked, 18-hemisphere and no-corrected-heatmap
variants are partial in all 40 analyses each, with at most 1, 4 and 5 regions within tolerance.
With the finer series, 434 of 440 are partial and 6 not reproduced. In the threshold notebook,
110 of 112 analyses at 14000 to 26000 are partial and 2 not reproduced (LRN lower in PACAPko).

## Summary

| Claim | Outcome |
|---|---|
| fig5-dg-fos | blocked: methods_underspecified (the readings of threshold, region size and outlier rule disagree; DG-sg and DG-po both significant in 12 of 20 at 4000 to 12000, in none at 14000 to 26000) |
| table4-lenient-regions | blocked: depends on fig5-dg-fos (partial under every reading at 4000 to 12000, not reproduced in 54 of 56 at 14000 to 26000) |
| table4-region-values | blocked: depends on fig5-dg-fos (partial under every reading at 4000 to 12000) |
| 24 claims on behaviour, qPCR, IHC, in situ hybridization, body weight and telemetry | blocked: data_missing |

How the analysis differs from the paper's: cells were recounted from the deposited binary
segmentations by the rule of the Supplemental Methods, with 26-connectivity assumed; the FRST
threshold the paper used is not known, so the five thresholds all hemispheres share were run here,
the finer series of 4 hemispheres as a variant, and 14000 to 26000 on 18 hemispheres in the
threshold notebook (28000 to 40000 were not run, for time); all 22 hemispheres were used where the
paper used 19 without saying which; 438 regions were tested where the paper tested 431, as the
paper's list of regions cannot be rebuilt from the dataset's label table; no region was excluded for
an artifact, as the dataset holds no artifact mask in the space of the counts.

What was found about the dataset: it holds only the whole-brain c-Fos experiment of the paper; it
holds 22 hemispheres of 13 mice where the paper used 19 of 12, without marking which; its count
tables cover 13 of the 22 hemispheres and count every connected component, not the 9 to 900 voxel
objects of the Methods (components of 9 to 900 voxels are 43% to 65% of their totals); the FRST
threshold used is not marked; 4 hemispheres were segmented on a different threshold series; the
corrected heatmaps of 2 hemispheres set 54467 and 1095526 voxels to 0, the only trace of an artifact
exclusion, in atlas space, for which the dataset holds no label image; 6 hemispheres have no masked
label image; the README describes the stacks as 25 um isotropic, while they are 3.7 x 3.7 x 5 um,
and links a different CATNIP repository and a 2024 preprint; the atlas-space heatmaps state
3.7 x 3.7 x 5 um voxels in their OME metadata and 25 um in their sidecar.

The main notebook took 66 minutes in this run, over the 60-minute limit; the recount cell alone
took 52 minutes.